In [21]:
%matplotlib inline
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display

In [ ]:
# ── Paths & mappings ────────────────────────────────────────────────────────
RESULTS_DIR = Path("../results/fold_eval_first_run")

MODEL_META = {
    "dcrnn_wind_dcrnn_base":     ("DCRNN",   "BASE"),
    "dcrnn_wind_dcrnn":          ("DCRNN",   "GRID"),
    "dcrnn_wind_dcrnn_nwp_hist": ("DCRNN",   "GRID+HIST"),
    "mtgnn_wind_mtgnn":          ("MTGNN",   "BASE"),
    "mtgnn_wind_mtgnn_nwp":      ("MTGNN",   "GRID"),
    "mtgnn_wind_mtgnn_nwp_hist": ("MTGNN",   "GRID+HIST"),
    "wavenet_wind_wavenet":      ("WaveNet", "BASE"),
}

FOLD_LABELS       = {0: "Aug\u2013Nov 2024", 1: "Nov 2024\u2013Mar 2025", 2: "Mar\u2013Aug 2025"}
FOLD_LABEL_TO_IDX = {v: k for k, v in FOLD_LABELS.items()}

# ── Model CSV loader ────────────────────────────────────────────────────────
def load_all_csvs():
    frames = []
    for path in sorted(RESULTS_DIR.glob("*.csv")):
        m = re.match(r"(.+)_fold(\d+)$", path.stem)
        if not m:
            continue
        prefix, fold = m.group(1), int(m.group(2))
        if prefix not in MODEL_META:
            continue
        model_name, variant = MODEL_META[prefix]
        df = pd.read_csv(path)
        df["model"]      = model_name
        df["variant"]    = variant
        df["fold"]       = fold
        df["fold_label"] = FOLD_LABELS[fold]
        frames.append(df)
    if not frames:
        print("No CSV files found in", RESULTS_DIR)
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

# ── Reference metrics loader ─────────────────────────────────────────────────
def load_reference_metrics():
    """Load reference_metrics.csv produced by evaluate_reference.py."""
    path = RESULTS_DIR / "reference_metrics.csv"
    if not path.exists():
        print(f"[reference] {path} not found — run evaluate_reference.py first.")
        print("  Reference bars in Plot 1 (MAE/R²) and Plot 2 (MAE/R²) won't be available.")
        return pd.DataFrame()
    df = pd.read_csv(path)
    df["fold_label"] = df["fold"].map(FOLD_LABELS)
    print(f"[reference] Loaded {len(df):,} rows ({df['station'].nunique()} stations, "
          f"{df['fold'].nunique()} folds, horizons {sorted(df['horizon'].unique())[:4]}…)")
    return df

data    = load_all_csvs()
ref_all = load_reference_metrics()

print(f"\nModel data: {len(data):,} rows | "
      f"{data['model'].nunique() if not data.empty else 0} models | "
      f"{data['fold'].nunique() if not data.empty else 0} folds")

[reference] Loaded 44,982 rows (153 stations, 3 folds, horizons [-1, 0, 1, 2]…)

Model data: 6,414 rows | 3 models | 3 folds


In [23]:
# ── Shared helpers ───────────────────────────────────────────────────────────

def ref_rmse_from_skill(df, ref_col):
    """Fallback: derive per-station reference RMSE from skill score."""
    denom = (1.0 - df[ref_col]).clip(lower=0.001, upper=0.999)
    return df["val_rmse"] / denom


def filter_model(df, variant, fold, scenario, seen):
    if variant == "BASE":
        mask = df["variant"] == "BASE"
    elif variant == "GRID":
        mask = (
            (df["model"].isin(["DCRNN", "MTGNN"]) & (df["variant"] == "GRID")) |
            (df["model"] == "WaveNet")
        )
    else:
        mask = (
            (df["model"].isin(["DCRNN", "MTGNN"]) & (df["variant"] == "GRID+HIST")) |
            (df["model"] == "WaveNet")
        )
    df = df[mask].copy()
    if fold != "Average":
        df = df[df["fold"] == FOLD_LABEL_TO_IDX[fold]]
    df = df[df["scenario"] == scenario]
    if seen == "Val stations only":
        df = df[df["seen"] == "n"]
    elif seen == "Train stations only":
        df = df[df["seen"] == "y"]
    return df


def filter_ref(df_ref, fold, seen, period="val", horizon=-1):
    """Filter reference_metrics to the aggregate val period for one fold."""
    if df_ref.empty:
        return df_ref
    df = df_ref[(df_ref["period"] == period) & (df_ref["horizon"] == horizon)].copy()
    if fold != "Average":
        df = df[df["fold"] == FOLD_LABEL_TO_IDX[fold]]
    if seen == "Val stations only":
        df = df[df["seen"] == "n"]
    elif seen == "Train stations only":
        df = df[df["seen"] == "y"]
    return df


def _bar_stats(series):
    return float(np.nanmean(series)), float(np.nanstd(series))

---
## Plot 1 — Model Comparison (Bar Chart)

In [24]:
def plot_bar(metric, variant, fold, scenario, seen):
    df  = filter_model(data, variant, fold, scenario, seen)
    ref = filter_ref(ref_all, fold, seen, period="val", horizon=-1)

    if df.empty:
        print("No data available for the current selection.")
        return

    MODELS = ["DCRNN", "MTGNN", "WaveNet"]
    bars, vals, errs, colors = [], [], [], []

    def _add_model(col):
        for m in MODELS:
            s = df[df["model"] == m][col]
            if s.empty: continue
            bars.append(m); v, e = _bar_stats(s); vals.append(v); errs.append(e)
            colors.append("steelblue")

    def _add_ref_from_ref_csv(i2_col, e2_col):
        """Add reference bars from reference_metrics.csv."""
        if ref.empty:
            return
        for col, label in [(i2_col, "ICON-D2"), (e2_col, "ECMWF")]:
            s = ref[col].dropna()
            if s.empty: continue
            bars.append(label); v, e = _bar_stats(s); vals.append(v); errs.append(e)
            colors.append("#888888")

    if metric == "RMSE":
        _add_model("val_rmse")
        if not ref.empty:
            _add_ref_from_ref_csv("icond2_rmse", "ecmwf_rmse")
        else:
            for ref_col, label in [("skill_icond2", "ICON-D2"), ("skill_ecmwf", "ECMWF")]:
                r = ref_rmse_from_skill(df, ref_col).dropna()
                if r.empty: continue
                bars.append(label); v, e = _bar_stats(r); vals.append(v); errs.append(e)
                colors.append("#888888")
        ylabel = "RMSE (m/s)"

    elif metric == "MAE":
        _add_model("val_mae")
        _add_ref_from_ref_csv("icond2_mae", "ecmwf_mae")
        ylabel = "MAE (m/s)"

    elif metric == "R\u00b2":
        _add_model("val_r2")
        _add_ref_from_ref_csv("icond2_r2", "ecmwf_r2")
        # Override colors for reference bars (already gray) — keep model bars green
        colors = ["#2ca02c" if c == "steelblue" else c for c in colors]
        ylabel = "R\u00b2"

    elif metric in ("Skill ICON-D2", "Skill ECMWF"):
        skill_col = "skill_icond2" if "ICON-D2" in metric else "skill_ecmwf"
        _add_model(skill_col)
        bars  += ["ICON-D2", "ECMWF"]; vals  += [0.0, 0.0]
        errs  += [0.0, 0.0];           colors += ["#888888", "#888888"]
        ylabel = f"Skill Score vs. {'ICON-D2' if 'ICON-D2' in metric else 'ECMWF'}"

    if not bars:
        print("No data available for the current selection.")
        return

    fig, ax = plt.subplots(figsize=(max(6, len(bars) * 1.3), 5))
    x = np.arange(len(bars))
    ax.bar(x, vals, yerr=errs, capsize=5, color=colors,
           edgecolor="white", linewidth=0.5,
           error_kw={"elinewidth": 1.5, "alpha": 0.7})

    if metric == "R\u00b2":
        ymin = min(min(v for v in vals if not np.isnan(v)) - 0.15, -0.05)
        ax.set_ylim(ymin, 1.12)
        ax.axhline(0.0, color="red", linestyle="--", linewidth=1.5, alpha=0.8)
        ax.axhline(1.0, color="red", linestyle="--", linewidth=1.5, alpha=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(bars, fontsize=12)
    ax.tick_params(axis="y", labelsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.axhline(0, color="gray", linewidth=0.6)

    fold_text = fold if fold != "Average" else "Average (all folds)"
    ax.text(
        0.98, 0.98,
        f"Fold: {fold_text}\nScenario: {scenario}\nStations: {seen}",
        transform=ax.transAxes, ha="right", va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#ccc", alpha=0.85),
    )
    plt.tight_layout()
    plt.show(); plt.close()


_style  = {"description_width": "70px"}
_layout = widgets.Layout(width="200px")

w1_metric   = widgets.Dropdown(
    options=["RMSE", "MAE", "R\u00b2", "Skill ICON-D2", "Skill ECMWF"],
    value="RMSE", description="Metric:", style=_style, layout=_layout)
w1_variant  = widgets.Dropdown(
    options=["BASE", "GRID", "GRID+HIST"],
    value="BASE", description="Variant:", style=_style, layout=_layout)
w1_fold     = widgets.Dropdown(
    options=list(FOLD_LABELS.values()) + ["Average"],
    value="Average", description="Fold:", style=_style, layout=_layout)
w1_scenario = widgets.Dropdown(
    options=["excl_val", "incl_val"],
    value="excl_val", description="Scenario:", style=_style, layout=_layout)
w1_seen     = widgets.Dropdown(
    options=["Val stations only", "Train stations only", "Both combined"],
    value="Val stations only", description="Stations:", style=_style, layout=_layout)

out1 = widgets.interactive_output(
    plot_bar,
    {"metric": w1_metric, "variant": w1_variant, "fold": w1_fold,
     "scenario": w1_scenario, "seen": w1_seen},
)
display(widgets.HBox([w1_metric, w1_variant, w1_fold, w1_scenario, w1_seen]), out1)

Output()

---
## Plot 2 — Station-Level Scatter (Model vs. Reference)

In [25]:
# Metric column mapping: (model_col, ref_i2_col, ref_e2_col, xlabel_base, higher_is_better)
_METRIC_MAP = {
    "RMSE": ("val_rmse", "icond2_rmse", "ecmwf_rmse", "RMSE (m/s)",  False),
    "MAE":  ("val_mae",  "icond2_mae",  "ecmwf_mae",  "MAE (m/s)",   False),
    "R²": ("val_r2", "icond2_r2", "ecmwf_r2",   "R²",       True),
}


def _get_ref_for_model(df_model, ref, model_col, ref_i2_col, ref_e2_col, fold):
    """
    Join model data with reference data on (station [, fold]).
    Falls back to skill-score-derived RMSE when reference CSV is missing.
    Returns df with columns: val_col, ref_i2, ref_e2.
    """
    if not ref.empty and ref_i2_col in ref.columns:
        merge_cols = ["station", "fold"] if fold == "Average" else ["station"]
        if fold == "Average":
            ref_agg = (
                ref.groupby("station")[[ref_i2_col, ref_e2_col]]
                .mean()
                .reset_index()
            )
            m_agg = (
                df_model.groupby("station")[[model_col]]
                .mean()
                .reset_index()
            )
            merged = m_agg.merge(ref_agg, on="station", how="inner")
        else:
            ref_sub = ref[["station", ref_i2_col, ref_e2_col]]
            merged  = df_model[["station", model_col]].merge(ref_sub, on="station", how="inner")
        merged = merged.rename(columns={ref_i2_col: "ref_i2", ref_e2_col: "ref_e2"})
        return merged[["station", model_col, "ref_i2", "ref_e2"]]

    # Fallback: only RMSE via skill scores
    if model_col == "val_rmse" and "skill_icond2" in df_model.columns:
        out = df_model[["station", model_col]].copy()
        out["ref_i2"] = ref_rmse_from_skill(df_model, "skill_icond2").values
        out["ref_e2"] = ref_rmse_from_skill(df_model, "skill_ecmwf").values
        if fold == "Average":
            out = out.groupby("station").mean().reset_index()
        return out

    return pd.DataFrame()


def _scatter_panel(ax, x, y, metric_name, xlabel):
    higher_better = _METRIC_MAP[metric_name][4]
    if higher_better:
        better = y > x   # higher R² → model better
    else:
        better = y < x   # lower RMSE/MAE → model better
    worse = ~better

    ax.scatter(x[better], y[better], c="#2ca02c", alpha=0.55, s=28,
               label=f"Better  (n={better.sum()})")
    ax.scatter(x[worse],  y[worse],  c="#d62728", alpha=0.55, s=28,
               label=f"Worse   (n={worse.sum()})")

    finite = np.concatenate([x, y])
    finite = finite[np.isfinite(finite)]
    if len(finite) == 0:
        return

    if higher_better:
        ax.plot([0, 1], [0, 1], "--", color="#aaa", linewidth=1.2, zorder=0)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
    else:
        lim_max = np.nanpercentile(finite, 99) * 1.08
        lim_max = max(lim_max, 0.3)
        ax.plot([0, lim_max], [0, lim_max], "--", color="#aaa",
                linewidth=1.2, zorder=0)
        ax.set_xlim(0, lim_max)
        ax.set_ylim(0, lim_max)

    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(f"Model {metric_name}", fontsize=12)
    ax.tick_params(labelsize=12)
    ax.legend(fontsize=10, loc="upper left", framealpha=0.85)


def plot_scatter(metric, model, variant, fold, scenario, seen):
    model_col, ref_i2_col, ref_e2_col, metric_unit, _ = _METRIC_MAP[metric]

    df  = filter_model(data, variant, fold, scenario, seen)
    ref = filter_ref(ref_all, fold, seen, period="val", horizon=-1)

    if df.empty:
        print("No model data for the current selection.")
        return

    df_m = df[df["model"] == model].copy()
    if df_m.empty:
        print(f"No data for model '{model}' with the current variant / fold selection.")
        return

    merged = _get_ref_for_model(df_m, ref, model_col, ref_i2_col, ref_e2_col, fold)
    if merged.empty:
        print(f"No reference data available for {metric}. "
              "Run evaluate_reference.py or select RMSE for skill-score fallback.")
        return

    merged = merged.dropna(subset=[model_col, "ref_i2", "ref_e2"])
    if merged.empty:
        print("All reference values are NaN.")
        return

    y = merged[model_col].values
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    for ax, ref_key, nwp_label in [
        (ax1, "ref_i2", "ICON-D2"),
        (ax2, "ref_e2", "ECMWF"),
    ]:
        x   = merged[ref_key].values
        ok  = np.isfinite(x) & np.isfinite(y)
        if ok.sum() < 2:
            ax.text(0.5, 0.5, f"No {nwp_label} reference data",
                    ha="center", va="center", transform=ax.transAxes, fontsize=12)
            continue
        _scatter_panel(ax, x[ok], y[ok], metric, f"{nwp_label} {metric_unit}")

    fold_text = fold if fold != "Average" else "Average (all folds)"
    fig.text(
        0.5, -0.02,
        f"Model: {model} | Variant: {variant} | Metric: {metric} | "
        f"Fold: {fold_text} | Scenario: {scenario} | Stations: {seen}",
        ha="center", fontsize=10, color="#555",
    )
    plt.tight_layout()
    plt.show(); plt.close()


_available_models = sorted(data["model"].unique()) if not data.empty else ["DCRNN", "MTGNN", "WaveNet"]

w2_metric   = widgets.Dropdown(
    options=["RMSE", "MAE", "R²"],
    value="RMSE", description="Metric:", style=_style, layout=_layout)
w2_model    = widgets.Dropdown(
    options=_available_models, value=_available_models[0],
    description="Model:", style=_style, layout=_layout)
w2_variant  = widgets.Dropdown(
    options=["BASE", "GRID", "GRID+HIST"],
    value="BASE", description="Variant:", style=_style, layout=_layout)
w2_fold     = widgets.Dropdown(
    options=list(FOLD_LABELS.values()) + ["Average"],
    value="Average", description="Fold:", style=_style, layout=_layout)
w2_scenario = widgets.Dropdown(
    options=["excl_val", "incl_val"],
    value="excl_val", description="Scenario:", style=_style, layout=_layout)
w2_seen     = widgets.Dropdown(
    options=["Val stations only", "Train stations only", "Both combined"],
    value="Val stations only", description="Stations:", style=_style, layout=_layout)

out2 = widgets.interactive_output(
    plot_scatter,
    {"metric": w2_metric, "model": w2_model, "variant": w2_variant,
     "fold": w2_fold, "scenario": w2_scenario, "seen": w2_seen},
)
display(widgets.HBox([w2_metric, w2_model, w2_variant, w2_fold, w2_scenario, w2_seen]), out2)

Output()

In [26]:
# ── Plot 3: Summary table — DCRNN vs MTGNN ──────────────────────────────────
_SEEN_TO_FILTER = {
    "Train": "Train stations only",
    "Val":   "Val stations only",
    "Both":  "Both combined",
}
_SEEN_TO_DATA = {"Train": "y", "Val": "n", "Both": None}


def show_summary_table(variant, seen):
    seen_val = _SEEN_TO_DATA[seen]

    mask = data["model"].isin(["DCRNN", "MTGNN"]) & (data["variant"] == variant)
    if seen_val is not None:
        mask &= data["seen"] == seen_val
    df_filt = data[mask]

    agg = (
        df_filt.groupby("model")[["val_r2", "val_rmse", "skill_icond2", "skill_ecmwf"]]
        .mean()
        .reindex(["DCRNN", "MTGNN"])
        .rename(columns={
            "val_r2":        "R²",
            "val_rmse":      "RMSE",
            "skill_icond2":  "Skill ICON-D2",
            "skill_ecmwf":   "Skill ECMWF",
        })
    )

    # Reference R² from NWP baselines (same value for both model rows)
    if not ref_all.empty and "icond2_r2" in ref_all.columns:
        ref_filt = filter_ref(ref_all, "Average", _SEEN_TO_FILTER[seen], period="val", horizon=-1)
        if not ref_filt.empty:
            agg["R² ICON-D2"] = ref_filt["icond2_r2"].mean()
            agg["R² ECMWF"]   = ref_filt["ecmwf_r2"].mean()

    grad_up   = [c for c in ["R²", "Skill ICON-D2", "Skill ECMWF"] if c in agg.columns]
    grad_down = [c for c in ["RMSE"] if c in agg.columns]

    styled = (
        agg.style
        .format("{:.4f}")
        .background_gradient(subset=grad_up,   cmap="RdYlGn",   axis=0)
        .background_gradient(subset=grad_down, cmap="RdYlGn_r", axis=0)
        .set_caption(f"Variant: {variant} | Seen: {seen} | Ø über alle Folds, Horizonte & Szenarien")
    )
    display(styled)


w3_variant = widgets.Dropdown(
    options=["BASE", "GRID", "GRID+HIST"],
    value="BASE", description="Variant:", style=_style, layout=_layout)
w3_seen = widgets.Dropdown(
    options=["Train", "Val", "Both"],
    value="Val", description="Seen:", style=_style, layout=_layout)

out3 = widgets.interactive_output(
    show_summary_table, {"variant": w3_variant, "seen": w3_seen})
display(widgets.HBox([w3_variant, w3_seen]), out3)

Output()

In [27]:
# ── Plot 3b: Comparison bar chart — ICON-D2 vs NWP only vs NWP + Hist ───────
def _improvement_bracket(ax, x0, x1, v0, v1, higher_better, lift=0.0,
                          color="dimgray", fontsize=11):
    """
    Significance bracket placed in the space between bar tops and ylim.
    `lift` = 0.0 places the bracket low in that space, 1.0 places it high.
    """
    if np.isnan(v0) or np.isnan(v1):
        return
    pct  = (v1 - v0) / abs(v0) * 100 if higher_better else (v0 - v1) / abs(v0) * 100
    sign = "+" if pct >= 0 else ""

    _, ymax    = ax.get_ylim()
    headroom   = ymax - max(v0, v1)        # space above the taller bar
    tick       = headroom * 0.10
    y_top      = max(v0, v1) + headroom * (0.28 + lift * 0.40)

    ax.plot([x0, x0, x1, x1],
            [v0 + tick, y_top, y_top, v1 + tick],
            color=color, lw=1.2)
    ax.text((x0 + x1) / 2, y_top + headroom * 0.05,
            f"{sign}{pct:.1f}%", ha="center", va="bottom",
            fontsize=fontsize, color=color, fontweight="bold")


def show_comparison_plot(seen):
    seen_val    = _SEEN_TO_DATA[seen]
    seen_filter = _SEEN_TO_FILTER[seen]

    # --- Reference (ICON-D2) ---
    ref_filt = filter_ref(ref_all, "Average", seen_filter, period="val", horizon=-1)
    r2_ref   = ref_filt["icond2_r2"].mean()   if (not ref_filt.empty and "icond2_r2"   in ref_filt.columns) else np.nan
    rmse_ref = ref_filt["icond2_rmse"].mean() if (not ref_filt.empty and "icond2_rmse" in ref_filt.columns) else np.nan

    # --- Model means (DCRNN GRID / MTGNN GRID+HIST, fixed) ---
    def _mean(model, variant, col):
        m = (data["model"] == model) & (data["variant"] == variant)
        if seen_val is not None:
            m &= data["seen"] == seen_val
        s = data[m][col]
        return float(s.mean()) if not s.empty else np.nan

    r2_vals   = [r2_ref,
                 _mean("DCRNN", "GRID",      "val_r2"),
                 _mean("MTGNN", "GRID+HIST", "val_r2")]
    rmse_vals = [rmse_ref,
                 _mean("DCRNN", "GRID",      "val_rmse"),
                 _mean("MTGNN", "GRID+HIST", "val_rmse")]

    labels = ["ICON-D2", "NWP only", "NWP + Hist"]
    green  = "#2ca02c"
    x      = np.arange(len(labels))
    bar_w  = 0.6

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # ── Left: R² ─────────────────────────────────────────────────────────────
    ax1.bar(x, r2_vals, color=green, alpha=0.85, width=bar_w, zorder=3)
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, fontsize=12)
    ax1.set_ylabel("R²", fontsize=12)
    ax1.tick_params(labelsize=12)
    ax1.set_ylim(0, 1.0)
    ax1.set_yticks(np.arange(0, 1.1, 0.2))
    ax1.set_yticklabels([f"{v:.1f}" for v in np.arange(0, 1.1, 0.2)], fontsize=12)

    # Dashed container outlines (0 → 1) per bar
    for xi in x:
        rect = mpatches.Rectangle(
            (xi - bar_w / 2, 0), bar_w, 1.0,
            fill=False, linestyle="--", edgecolor="gray",
            linewidth=0.9, alpha=0.65, zorder=4,
        )
        ax1.add_patch(rect)

    # Brackets inside the headroom between bar tops and y=1
    _improvement_bracket(ax1, 0, 1, r2_vals[0], r2_vals[1], higher_better=True, lift=0.0)
    _improvement_bracket(ax1, 0, 2, r2_vals[0], r2_vals[2], higher_better=True, lift=1.0)

    ax1.set_title("R²", fontsize=12)

    # ── Right: RMSE ──────────────────────────────────────────────────────────
    ax2.bar(x, rmse_vals, color=green, alpha=0.85, width=bar_w, zorder=3)
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels, fontsize=12)
    ax2.set_ylabel("RMSE (m/s)", fontsize=12)
    ax2.tick_params(labelsize=12)

    max_rmse = max((v for v in rmse_vals if not np.isnan(v)), default=1.0)
    ax2.set_ylim(0, max_rmse * 1.45)

    _improvement_bracket(ax2, 0, 1, rmse_vals[0], rmse_vals[1], higher_better=False, lift=0.0)
    _improvement_bracket(ax2, 0, 2, rmse_vals[0], rmse_vals[2], higher_better=False, lift=1.0)

    ax2.set_title("RMSE", fontsize=12)

    fig.suptitle(f"NWP baseline comparison — Stations: {seen}", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()
    plt.close()


out_cmp = widgets.interactive_output(show_comparison_plot, {"seen": w3_seen})
display(out_cmp)

Output()

In [28]:
# ── Raw predictions loader ────────────────────────────────────────────────────
RAW_DIR = Path("../data/raw_preds")

def load_raw_preds():
    frames = []
    for path in sorted(RAW_DIR.glob("*_raw.parquet")):
        # Filename: {prefix}_fold{n}_raw.parquet
        stem = path.stem[:-4]  # strip _raw suffix (4 chars)
        m = re.match(r"(.+)_fold(\d+)$", stem)
        if not m:
            continue
        prefix = m.group(1)
        fold   = int(m.group(2))
        if prefix not in MODEL_META:
            continue
        model_name, variant = MODEL_META[prefix]
        df = pd.read_parquet(path)
        df["model"]      = model_name
        df["variant"]    = variant
        df["fold"]       = fold
        df["fold_label"] = FOLD_LABELS.get(fold, "?")
        df["hour"]       = pd.to_datetime(df["valid_time"]).dt.hour
        df["month"]      = pd.to_datetime(df["valid_time"]).dt.month
        frames.append(df)
    if not frames:
        print(f"Keine Rohdaten in {RAW_DIR} — bitte get_test_results_*.py ausführen.")
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

raw = load_raw_preds()
print(f"Raw predictions: {len(raw):,} Zeilen | {raw['model'].nunique() if not raw.empty else 0} Modelle")


Keine Rohdaten in ../data/raw_preds — bitte get_test_results_*.py ausführen.
Raw predictions: 0 Zeilen | 0 Modelle


---
## Plot 4 — Fehler nach Prognosehorizont

In [29]:
# ── Shared helper for raw stratification ─────────────────────────────────────
_MODEL_COLORS = {"DCRNN": "steelblue", "MTGNN": "darkorange", "WaveNet": "forestgreen"}

def _compute_metric(grp, metric):
    err = grp["pred"] - grp["gt"]
    if metric == "RMSE":
        return float(np.sqrt((err ** 2).mean()))
    elif metric == "MAE":
        return float(err.abs().mean())
    else:
        raise ValueError(f"Unknown metric {metric}")

def _compute_metric_ref(grp, metric, col):
    err = grp[col] - grp["gt"]
    if metric == "RMSE":
        return float(np.sqrt((err ** 2).mean()))
    return float(err.abs().mean())


def plot_horizon(metric, variant, fold):
    if raw.empty:
        print("Keine Rohdaten vorhanden.")
        return

    df = raw.copy()
    if variant != "All":
        df = df[df["variant"] == variant]
    if fold != "Average":
        df = df[df["fold"] == FOLD_LABEL_TO_IDX[fold]]
    if df.empty:
        print("Keine Daten für diese Auswahl.")
        return

    fig, ax = plt.subplots(figsize=(12, 5))

    for mdl in ["DCRNN", "MTGNN", "WaveNet"]:
        dm = df[df["model"] == mdl]
        if dm.empty:
            continue
        vals = dm.groupby("horizon").apply(lambda g: _compute_metric(g, metric))
        ax.plot(vals.index, vals.values, label=mdl, color=_MODEL_COLORS.get(mdl),
                linewidth=2, marker=".", markersize=4)

    # NWP baseline (ICON-D2 nearest grid point)
    if "nwp_ref" in df.columns:
        nwp_vals = df.groupby("horizon").apply(lambda g: _compute_metric_ref(g, metric, "nwp_ref"))
        ax.plot(nwp_vals.index, nwp_vals.values, label="ICON-D2", color="#888",
                linewidth=1.5, linestyle="--")

    ax.set_xlabel("Prognosehorizont (Stunden)", fontsize=12)
    ax.set_ylabel(f"{metric} (m/s)", fontsize=12)
    ax.legend(fontsize=11, loc="upper left")
    ax.grid(axis="y", alpha=0.4)
    fold_text = fold if fold != "Average" else "Ø alle Folds"
    ax.set_title(f"{metric} nach Prognosehorizont | Variante: {variant} | {fold_text}", fontsize=13)
    plt.tight_layout()
    plt.show(); plt.close()


_style4  = {"description_width": "80px"}
_layout4 = widgets.Layout(width="210px")

w4_metric  = widgets.Dropdown(options=["RMSE", "MAE"], value="RMSE",
                              description="Metrik:", style=_style4, layout=_layout4)
w4_variant = widgets.Dropdown(options=["All", "BASE", "GRID", "GRID+HIST"], value="BASE",
                              description="Variante:", style=_style4, layout=_layout4)
w4_fold    = widgets.Dropdown(options=list(FOLD_LABELS.values()) + ["Average"], value="Average",
                              description="Fold:", style=_style4, layout=_layout4)

out4 = widgets.interactive_output(
    plot_horizon, {"metric": w4_metric, "variant": w4_variant, "fold": w4_fold}
)
display(widgets.HBox([w4_metric, w4_variant, w4_fold]), out4)


Output()

---
## Plot 5 — Fehler nach Windgeschwindigkeit / Tagesstunde / Monat / Horizont (Boxplots)

In [30]:
# ── Plot 5 helper ────────────────────────────────────────────────────────────
_WS_BINS = list(range(0, 22, 2))   # 0-2, 2-4, ..., 18-20, 20+
_WS_LABELS = [f"{_WS_BINS[i]}-{_WS_BINS[i+1]}" for i in range(len(_WS_BINS) - 1)]

_X_OPTS = {
    "Prognosehorizont": ("horizon",  "Prognosehorizont (h)"),
    "Tagesstunde":      ("hour",     "Stunde (UTC)"),
    "Monat":            ("month",    "Monat"),
    "WS-Klasse (m/s)":  ("ws_bin",  "Windgeschwindigkeit (m/s)"),
}


def plot_stratified(x_axis, metric, variant, fold, plot_type):
    if raw.empty:
        print("Keine Rohdaten vorhanden.")
        return

    df = raw.copy()
    if variant != "All":
        df = df[df["variant"] == variant]
    if fold != "Average":
        df = df[df["fold"] == FOLD_LABEL_TO_IDX[fold]]
    if df.empty:
        print("Keine Daten für diese Auswahl.")
        return

    x_col, xlabel = _X_OPTS[x_axis]

    # Build x-axis grouping column
    if x_col == "ws_bin":
        df = df.copy()
        df["ws_bin"] = pd.cut(df["gt"], bins=_WS_BINS + [999], labels=_WS_LABELS, right=False)
        df = df.dropna(subset=["ws_bin"])

    models = [m for m in ["DCRNN", "MTGNN", "WaveNet"] if m in df["model"].unique()]
    if not models:
        print("Keine Modell-Daten für diese Auswahl.")
        return

    if plot_type == "Linie":
        fig, ax = plt.subplots(figsize=(12, 5))
        for mdl in models:
            dm = df[df["model"] == mdl]
            vals = dm.groupby(x_col).apply(lambda g: _compute_metric(g, metric))
            ax.plot(
                range(len(vals)) if x_col == "ws_bin" else vals.index,
                vals.values, label=mdl, color=_MODEL_COLORS.get(mdl), linewidth=2, marker="."
            )
        if x_col == "ws_bin":
            ax.set_xticks(range(len(_WS_LABELS)))
            ax.set_xticklabels(_WS_LABELS, rotation=45, ha="right")
        ax.set_xlabel(xlabel, fontsize=12)
        ax.set_ylabel(f"{metric} (m/s)", fontsize=12)
        ax.legend(fontsize=11)
        ax.grid(axis="y", alpha=0.4)

    else:  # Boxplot — Fehler abs(pred - gt) pro x-Klasse
        n_models = len(models)
        fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5), sharey=True)
        if n_models == 1:
            axes = [axes]
        for ax, mdl in zip(axes, models):
            dm = df[df["model"] == mdl].copy()
            dm["abs_err"] = (dm["pred"] - dm["gt"]).abs()
            if x_col == "ws_bin":
                order = _WS_LABELS
            else:
                order = sorted(dm[x_col].unique())
            groups = [dm[dm[x_col] == v]["abs_err"].dropna().values for v in order]
            ax.boxplot(groups, labels=[str(v) for v in order],
                       patch_artist=True,
                       boxprops=dict(facecolor=_MODEL_COLORS.get(mdl, "steelblue"), alpha=0.6))
            ax.set_title(mdl, fontsize=12)
            ax.set_xlabel(xlabel, fontsize=11)
            ax.tick_params(axis="x", rotation=45)
            if ax == axes[0]:
                ax.set_ylabel("|Fehler| (m/s)", fontsize=11)
            ax.grid(axis="y", alpha=0.3)

    fold_text = fold if fold != "Average" else "Ø alle Folds"
    fig.suptitle(
        f"{x_axis} | Metrik: {metric} | Variante: {variant} | {fold_text}",
        fontsize=13, y=1.02
    )
    plt.tight_layout()
    plt.show(); plt.close()


_style5  = {"description_width": "90px"}
_layout5 = widgets.Layout(width="220px")

w5_xaxis   = widgets.Dropdown(options=list(_X_OPTS.keys()), value="WS-Klasse (m/s)",
                               description="X-Achse:", style=_style5, layout=_layout5)
w5_metric  = widgets.Dropdown(options=["RMSE", "MAE"], value="RMSE",
                               description="Metrik:", style=_style5, layout=_layout5)
w5_variant = widgets.Dropdown(options=["All", "BASE", "GRID", "GRID+HIST"], value="BASE",
                               description="Variante:", style=_style5, layout=_layout5)
w5_fold    = widgets.Dropdown(options=list(FOLD_LABELS.values()) + ["Average"], value="Average",
                               description="Fold:", style=_style5, layout=_layout5)
w5_type    = widgets.Dropdown(options=["Boxplot", "Linie"], value="Boxplot",
                               description="Plottyp:", style=_style5, layout=_layout5)

out5 = widgets.interactive_output(
    plot_stratified,
    {"x_axis": w5_xaxis, "metric": w5_metric, "variant": w5_variant,
     "fold": w5_fold, "plot_type": w5_type}
)
display(widgets.HBox([w5_xaxis, w5_metric, w5_variant, w5_fold, w5_type]), out5)


Output()